In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import os
import copy
import time
from google.colab import drive

In [2]:
# 1. MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 2. DEFINE PATH & CONFIGURATION
# IMPORTANT: Ensure this path points to the folder containing 'cancer' and 'non cancer'
data_dir = '/content/drive/MyDrive/OC Dataset'
batch_size = 32
num_epochs = 15
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [5]:
# 3. DATA AUGMENTATION & NORMALIZATION
# Resizing to 300x300 for EfficientNet-B3
data_transforms = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [7]:
# 4. LOAD DATA AND SPLIT
full_dataset = datasets.ImageFolder(data_dir, transform=data_transforms)

# Splitting 80% for training, 20% for validation
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
    'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
}

dataset_sizes = {'train': train_size, 'val': val_size}
class_names = full_dataset.classes
print(f"Classes found: {class_names}")
print(f"Training images: {train_size}, Validation images: {val_size}")

Classes found: ['CANCER', 'NON CANCER']
Training images: 760, Validation images: 190


In [9]:
# 5. INITIALIZE EFFICIENTNET-B3 MODEL
print("Initializing EfficientNet-B3...")
model = models.efficientnet_b3(weights='DEFAULT')

# Adjust the final classifier layer for 2 classes
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, len(class_names))

model = model.to(device)

Initializing EfficientNet-B3...


In [10]:
# 6. DEFINE LOSS AND OPTIMIZER
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [11]:
# 7. TRAINING FUNCTION
def train_model(model, criterion, optimizer, num_epochs=15):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch + 1}/{num_epochs}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')

    model.load_state_dict(best_model_wts)
    return model

In [12]:
# 8. START TRAINING
model_ft = train_model(model, criterion, optimizer, num_epochs=num_epochs)

Epoch 1/15
----------


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


train Loss: 0.5792 Acc: 0.7316
val Loss: 0.4615 Acc: 0.8526

Epoch 2/15
----------
train Loss: 0.3299 Acc: 0.8987
val Loss: 0.2930 Acc: 0.8895

Epoch 3/15
----------
train Loss: 0.1784 Acc: 0.9461
val Loss: 0.2261 Acc: 0.9158

Epoch 4/15
----------
train Loss: 0.1128 Acc: 0.9592
val Loss: 0.2089 Acc: 0.9263

Epoch 5/15
----------
train Loss: 0.0651 Acc: 0.9868
val Loss: 0.2194 Acc: 0.9211

Epoch 6/15
----------
train Loss: 0.0574 Acc: 0.9855
val Loss: 0.2201 Acc: 0.9263

Epoch 7/15
----------
train Loss: 0.0310 Acc: 0.9921
val Loss: 0.2207 Acc: 0.9316

Epoch 8/15
----------
train Loss: 0.0280 Acc: 0.9921
val Loss: 0.2196 Acc: 0.9474

Epoch 9/15
----------
train Loss: 0.0267 Acc: 0.9921
val Loss: 0.2219 Acc: 0.9316

Epoch 10/15
----------
train Loss: 0.0134 Acc: 0.9947
val Loss: 0.2112 Acc: 0.9579

Epoch 11/15
----------
train Loss: 0.0125 Acc: 0.9974
val Loss: 0.2006 Acc: 0.9421

Epoch 12/15
----------
train Loss: 0.0164 Acc: 0.9947
val Loss: 0.2425 Acc: 0.9474

Epoch 13/15
----------


In [13]:
# 9. SAVE THE MODEL FOR YOUR WEBSITE
save_path = '/content/drive/MyDrive/OC Dataset/oral_cancer_model.pth'
torch.save(model_ft.state_dict(), save_path)
print(f"Model saved successfully at: {save_path}")

Model saved successfully at: /content/drive/MyDrive/OC Dataset/oral_cancer_model.pth
